In [ ]:
import requests
import pandas as pd
import time
import os
import json

NameError: name 'pd' is not defined

In [ ]:

# ── Config ────────────────────────────────────────────────────────────────────
APP_TOKEN  = "i0HPWlhXINORN4CrngRTlldoX"
BASE_URL   = "https://data.cityofchicago.org/resource/ijzp-q8t2.json"
BATCH_SIZE = 50000
OUTPUT_DIR = "chicago_crime_data"
PROGRESS_FILE = os.path.join(OUTPUT_DIR, "progress.json")
FINAL_OUTPUT  = os.path.join(OUTPUT_DIR, "chicago_crimes_2001_2025.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Progress helpers ──────────────────────────────────────────────────────────
def load_progress():
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE, "r") as f:
            return json.load(f)
    return {"offset": 0, "batch_num": 1, "total_records": 0}

def save_progress(offset, batch_num, total_records):
    with open(PROGRESS_FILE, "w") as f:
        json.dump({
            "offset": offset,
            "batch_num": batch_num,
            "total_records": total_records
        }, f)

# ── Fetch ─────────────────────────────────────────────────────────────────────
def fetch_batch(offset, retries=3):
    params = {
        "$limit":  BATCH_SIZE,
        "$offset": offset,
        "$order":  "date ASC",
        "$$app_token": APP_TOKEN
    }
    for attempt in range(1, retries + 1):
        try:
            response = requests.get(BASE_URL, params=params, timeout=60)
            response.raise_for_status()
            return response.json()
        except requests.RequestException as e:
            print(f"  [Attempt {attempt}/{retries}] Request failed: {e}")
            if attempt < retries:
                time.sleep(5 * attempt)  # Exponential backoff
            else:
                raise

# ── Main download ─────────────────────────────────────────────────────────────
def download_all():
    progress  = load_progress()
    offset    = progress["offset"]
    batch_num = progress["batch_num"]
    total     = progress["total_records"]

    if offset > 0:
        print(f"Resuming from batch {batch_num} (offset={offset}, {total} records already saved)\n")
    else:
        print("Starting fresh download...\n")

    # Write CSV header only on first run
    write_header = not os.path.exists(FINAL_OUTPUT) or offset == 0
    file_mode = "w" if write_header else "a"

    with open(FINAL_OUTPUT, file_mode, encoding="utf-8") as csv_file:
        while True:
            print(f"Fetching batch {batch_num} (offset={offset})...")

            batch = fetch_batch(offset)

            if not batch:
                print("No more data. Download complete.")
                break

            batch_df = pd.DataFrame(batch)

            # Append to CSV (header only on first write)
            batch_df.to_csv(csv_file, index=False, header=write_header)
            write_header = False  # Only write header once

            total     += len(batch)
            offset    += BATCH_SIZE
            batch_num += 1

            save_progress(offset, batch_num, total)
            print(f"  -> {len(batch)} records fetched. Total so far: {total}")

            time.sleep(0.5)

    # Clean up progress file on successful completion
    if os.path.exists(PROGRESS_FILE):
        os.remove(PROGRESS_FILE)
        print("Progress file removed.")

    print(f"\nDone! Total records: {total}")
    print(f"Saved to: {FINAL_OUTPUT}")

# ── Entry point ───────────────────────────────────────────────────────────────
if __name__ == "__main__":
    download_all()
    

In [ ]:
df = pd.read_csv(FINAL_OUTPUT)


In [ ]:
print(df.shape)
print(df.dtypes)


In [ ]:
df.head()
df.isnull().sum()


In [ ]:
#convert date and updated_on to datetime, errrors="coerce" will set invalid parsing to NaT, as this treats null values and malformed date strings the same (Nat, not a time), reduced likelihood of value error that prevents algo from running
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["updated_on"] = pd.to_datetime(df["updated_on"], errors="coerce")


In [ ]:
#inspecting, 0 so no need to change anything
df[df['date'].isna()]
df[df['updated_on'].isna()]

In [ ]:
#this has changed massively since being recoded, meaning it is probably formatted very inconsistently
#lets explore, this is all ISO 8601 format
df_raw = pd.read_csv(FINAL_OUTPUT, low_memory=False)
df_raw['updated_on'].dropna().unique()[:20]
df['updated_on'] = pd.to_datetime(df['updated_on'], format='%Y-%m-%dT%H:%M:%S.%f', errors='coerce')
#still massive, meaning lots of empty were read as string, " ", so lets double check
# df['updated_on'].isna().sum()


In [ ]:
# Check for empty strings
print("Empty strings:", (df_raw['updated_on'] == '').sum())

# Check for actual nulls
print("Null values:", df_raw['updated_on'].isna().sum())

# Check for actual values
print("Has value:", df_raw['updated_on'].notna().sum() - (df_raw['updated_on'] == '').sum())
# Find values that failed to parse, these are numeric ids, meaning CSV probably got misalligned, 
mask = df['updated_on'].isna() & df_raw['updated_on'].notna()
df_raw.loc[mask, 'updated_on'].unique()[:20]
#lets see what else got misalligned
mask = df['updated_on'].isna() & df_raw['updated_on'].notna()
print(mask.sum())
df_raw.loc[mask].head(3)
print(df_raw.columns.tolist())
print(df_raw.shape)
df_raw.loc[mask, ['id', 'year', 'updated_on', 'x_coordinate', 'date']].head(10)
df.loc[mask, 'year'].value_counts().sort_index()
#the fact that corruption is consistent among all years implies the issue is with the download process
# Check if corrupted rows are clustered or random
mask_indices = df_raw.index[mask].tolist()
# Check gaps between corrupted row indices
import numpy as np
gaps = np.diff(mask_indices[:1000])
print("Min gap:", gaps.min())
print("Max gap:", gaps.max())
print("Mean gap:", gaps.mean())
print("First corrupted row:", mask_indices[0])
print("Last corrupted row:", mask_indices[-1])
print("Total corrupted:", len(mask_indices))
df_raw.iloc[649998:650002][['id', 'year', 'updated_on', 'x_coordinate']]
df_raw2 = pd.read_csv(FINAL_OUTPUT, low_memory=False)
df_raw2.iloc[649998:650002][['id', 'date', 'year', 'updated_on', 'x_coordinate', 'y_coordinate', 'latitude', 'longitude']]
# Check the transition point in raw data
df_raw.iloc[649998:650002][['id', 'date', 'year', 'updated_on', 'x_coordinate', 'y_coordinate', 'latitude', 'longitude']]
#so there is a download errorfrom the 2nd-13th batch, so I'm going to download differerntly
df_clean = df_raw.iloc[:650000]
df_clean.to_csv(FINAL_OUTPUT, index=False)
print(f"Clean rows saved: {len(df_clean)}")
print(f"Last clean date: {df_clean['date'].iloc[-1]}")
# Check what's in each column for corrupted rows
df_raw.iloc[650000][['year', 'updated_on', 'x_coordinate', 'y_coordinate', 'latitude', 'longitude']]
df_raw.iloc[1000000][['year', 'updated_on', 'x_coordinate', 'y_coordinate']]
# Split into clean and corrupted
df_clean = df_raw.iloc[:650000].copy()
df_corrupted = df_raw.iloc[650000:].copy()

# Fix the column shift
df_corrupted['year']         = df_corrupted['x_coordinate']
df_corrupted['updated_on']   = df_corrupted['y_coordinate']
df_corrupted['x_coordinate'] = df_corrupted['updated_on']
df_corrupted['y_coordinate'] = df_corrupted['year']

# Recombine
df_fixed = pd.concat([df_clean, df_corrupted], ignore_index=True)

# Verify the fix at the boundary
df_fixed.iloc[649998:650002][['id', 'date', 'year', 'updated_on', 'x_coordinate', 'y_coordinate']]
df_clean = df_raw.iloc[:650000].copy()
df_corrupted = df_raw.iloc[650000:].copy()

# Use temp variables to avoid circular overwrite
temp_year         = df_corrupted['x_coordinate'].copy()
temp_updated_on   = df_corrupted['y_coordinate'].copy()
temp_x_coordinate = df_corrupted['year'].copy()
temp_y_coordinate = df_corrupted['updated_on'].copy()

df_corrupted['year']         = temp_year
df_corrupted['updated_on']   = temp_updated_on
df_corrupted['x_coordinate'] = temp_x_coordinate
df_corrupted['y_coordinate'] = temp_y_coordinate

# Recombine
df_fixed = pd.concat([df_clean, df_corrupted], ignore_index=True)

# Verify
df_fixed.iloc[649998:650002][['id', 'date', 'year', 'updated_on', 'x_coordinate', 'y_coordinate']]
df_raw.iloc[650000].to_dict()
df_clean = df_raw.iloc[:650000].copy()
df_corrupted = df_raw.iloc[650000:].copy()

temp_x         = df_corrupted['year'].copy()        # x_coordinate is in year
temp_y         = df_corrupted['updated_on'].copy()  # y_coordinate is in updated_on
temp_year      = df_corrupted['x_coordinate'].copy() # year is in x_coordinate
temp_updated   = df_corrupted['y_coordinate'].copy() # updated_on is in y_coordinate

df_corrupted['x_coordinate'] = temp_x
df_corrupted['y_coordinate'] = temp_y
df_corrupted['year']         = temp_year
df_corrupted['updated_on']   = temp_updated

# Recombine
df_fixed = pd.concat([df_clean, df_corrupted], ignore_index=True)

# Verify
df_fixed.iloc[649998:650002][['id', 'date', 'year', 'updated_on', 'x_coordinate', 'y_coordinate']]
print(FINAL_OUTPUT)
print(os.listdir(OUTPUT_DIR))
import os
os.rename(
    'chicago_crime_data/chicago_crimes_2001_2025.csv',
    'chicago_crime_data/chicago_crimes_CORRUPTED.csv'
)
df_corrupted = pd.read_csv('chicago_crime_data/chicago_crimes_CORRUPTED.csv', low_memory=False)
print(df_corrupted.iloc[0].to_dict())
# Find first row where year is not a valid year
invalid_year = pd.to_numeric(df_corrupted['year'], errors='coerce')
first_bad = df_corrupted[invalid_year > 9999].index[0]
print(f"First corrupted row: {first_bad}")
print(df_corrupted.iloc[first_bad].to_dict())
print(df_corrupted.iloc[first_bad - 1].to_dict())
import requests
import pandas as pd
import os
import time

APP_TOKEN    = "i0HPWlhXINORN4CrngRTlldoX"
BASE_URL     = "https://data.cityofchicago.org/resource/ijzp-q8t2.json"
BATCH_SIZE   = 50000
OUTPUT_DIR   = "chicago_crime_data"
BATCH_DIR    = os.path.join(OUTPUT_DIR, "batches")
FINAL_OUTPUT = os.path.join(OUTPUT_DIR, "chicago_crimes_2001_2025.csv")

os.makedirs(BATCH_DIR, exist_ok=True)

EXPECTED_COLUMNS = [
    'id', 'case_number', 'date', 'block', 'iucr', 'primary_type',
    'description', 'location_description', 'arrest', 'domestic', 'beat',
    'district', 'ward', 'community_area', 'fbi_code', 'year', 'updated_on',
    'x_coordinate', 'y_coordinate', 'latitude', 'longitude', 'location'
]

def validate_batch(batch_df, batch_num):
    years = pd.to_numeric(batch_df['year'], errors='coerce').dropna()
    if (years > 2030).any() or (years < 2000).any():
        raise ValueError(f"Batch {batch_num}: suspicious year values detected")
    x_coords = pd.to_numeric(batch_df['x_coordinate'], errors='coerce').dropna()
    if (x_coords > 1200000).any():
        raise ValueError(f"Batch {batch_num}: x_coordinate has suspiciously large values")
    return True

def fetch_batch(offset, retries=3):
    params = {
        "$limit":      BATCH_SIZE,
        "$offset":     offset,
        "$order":      "date ASC",
        "$$app_token": APP_TOKEN
    }
    for attempt in range(1, retries + 1):
        try:
            response = requests.get(BASE_URL, params=params, timeout=60)
            response.raise_for_status()
            return response.json()
        except requests.RequestException as e:
            print(f"  [Attempt {attempt}/{retries}] Request failed: {e}")
            if attempt < retries:
                time.sleep(5 * attempt)
            else:
                raise

def download_all():
    offset    = 0
    batch_num = 1
    total     = 0

    print("Starting full download...\n")

    while True:
        batch_file = os.path.join(BATCH_DIR, f"batch_{batch_num:04d}.csv")

        # Skip already downloaded batches (allows resuming)
        if os.path.exists(batch_file):
            print(f"Batch {batch_num} already exists, skipping...")
            offset    += BATCH_SIZE
            batch_num += 1
            continue

        print(f"Fetching batch {batch_num} (offset={offset})...")

        batch = fetch_batch(offset)

        if not batch:
            print("No more data. Download complete.")
            break

        batch_df = pd.DataFrame(batch)

        # Ensure consistent columns
        for col in EXPECTED_COLUMNS:
            if col not in batch_df.columns:
                batch_df[col] = None
        batch_df = batch_df[EXPECTED_COLUMNS]

        # Validate before saving
        try:
            validate_batch(batch_df, batch_num)
        except ValueError as e:
            print(f"  !! Validation failed: {e}")
            print(f"  Stopping at batch {batch_num} to prevent corruption.")
            break

        # Save individual batch file
        batch_df.to_csv(batch_file, index=False)

        total     += len(batch)
        offset    += BATCH_SIZE
        batch_num += 1

        print(f"  -> {len(batch)} records fetched. Total so far: {total}")
        time.sleep(0.5)

    print(f"\nDone! Total records: {total}")

def combine_batches():
    print("Combining batches into final CSV...")
    batch_files = sorted([
        os.path.join(BATCH_DIR, f) 
        for f in os.listdir(BATCH_DIR) 
        if f.endswith('.csv')
    ])

    dfs = []
    for f in batch_files:
        dfs.append(pd.read_csv(f, low_memory=False))

    df_final = pd.concat(dfs, ignore_index=True)
    df_final.to_csv(FINAL_OUTPUT, index=False)
    print(f"Saved {len(df_final)} rows to {FINAL_OUTPUT}")

# ── Entry point ───────────────────────────────────────────────────────────────
download_all()
combine_batches()
download_all()